<a href="https://colab.research.google.com/github/irfan060606/Scikit-Learn-Cookbook/blob/main/Chapter_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 — Common Conventions and API Elements of scikit-learn

**Buku acuan:** *scikit-learn Cookbook*, Third Edition — John Sukup (Packt, 2025)  
**Fokus:** memahami pola API scikit-learn dan menerapkannya dalam eksperimen Python yang dapat dijalankan ulang.

Notebook ini disusun sebagai catatan belajar mandiri berdasarkan topik-topik Chapter 1. Contoh data dan eksperimen di notebook ini disusun khusus untuk latihan; hasil angka dapat berbeda bila konfigurasi atau versi pustaka berubah.

## Tujuan pembelajaran

Setelah menyelesaikan notebook ini, saya diharapkan mampu:

1. Menjelaskan alasan scikit-learn menggunakan antarmuka yang konsisten dan modular.
2. Membedakan estimator, predictor, dan transformer.
3. Memahami perbedaan `fit()`, `predict()`, `transform()`, `fit_transform()`, serta `fit_predict()`.
4. Membuat transformer sederhana yang mengikuti pola API scikit-learn.
5. Menggabungkan preprocessing dan model dengan `Pipeline`.
6. Membaca atribut hasil training dan mengatur hyperparameter.
7. Memilih hyperparameter menggunakan pencarian berbasis cross-validation.
8. Mengenali fungsi metadata tags dan menerapkan praktik penggunaan API yang baik.

## 1. Persiapan lingkungan

Chapter ini menggunakan Python, NumPy, dan scikit-learn. Bila menjalankan notebook di Google Colab, biasanya NumPy dan scikit-learn sudah tersedia. Sel kode berikut memeriksa versi pustaka yang aktif.

In [ ]:
import numpy as np
import sklearn

print("Python package scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)

Python package scikit-learn: 1.6.1
NumPy: 2.1.3


## 2. Filosofi desain scikit-learn

scikit-learn mengutamakan **konsistensi, modularitas, kesederhanaan, dan penggunaan ulang**. Banyak objek mengikuti pola yang mirip sehingga kita tidak perlu mempelajari cara penggunaan yang benar-benar berbeda untuk setiap algoritma.

- **Estimator** adalah objek yang mempelajari pola atau parameter dari data.
- `fit()` digunakan untuk mempelajari data.
- `predict()` menghasilkan prediksi dari model yang sudah dilatih.
- **Transformer** mempelajari aturan perubahan data dan menerapkannya melalui `transform()`.
- **Pipeline** menghubungkan beberapa tahapan menjadi satu alur kerja.

Pola ini mempermudah eksperimen, penggantian algoritma, dan pengulangan proses secara konsisten.

## 3. Memahami estimator: `fit()` dan `predict()`

Untuk melihat pola estimator tanpa bergantung pada dataset eksternal, kita gunakan contoh sederhana tentang perkiraan nilai latihan. Dua fitur masukan adalah jumlah jam belajar dan jumlah latihan yang dikerjakan. Target `nilai` merupakan data contoh buatan, bukan data mahasiswa sungguhan.

Dalam supervised learning, `fit(X, y)` mempelajari hubungan antara fitur `X` dan target `y`. Setelah itu, `predict(X_baru)` memperkirakan target untuk data yang belum diberikan saat pelatihan.

In [ ]:
from sklearn.linear_model import LinearRegression

# Setiap baris: [jam belajar per hari, jumlah latihan yang dikerjakan]
X_latihan = np.array([
    [1.0, 2],
    [1.5, 3],
    [2.0, 3],
    [2.5, 5],
    [3.0, 5],
    [3.5, 6],
    [4.0, 7],
    [4.5, 8],
])
nilai = np.array([52, 57, 61, 68, 72, 76, 82, 87])

regressor = LinearRegression()
regressor.fit(X_latihan, nilai)

skenario_baru = np.array([
    [2.0, 4],
    [4.0, 6],
])
prediksi_nilai = regressor.predict(skenario_baru)

print("Prediksi untuk [jam belajar, jumlah latihan]:")
for fitur, hasil in zip(skenario_baru, prediksi_nilai):
    print(f"{fitur.tolist()} -> perkiraan nilai {hasil:.1f}")

Prediksi untuk [jam belajar, jumlah latihan]:
[2.0, 4.0] -> perkiraan nilai 62.5
[4.0, 6.0] -> perkiraan nilai 80.4


### Interpretasi

Objek `regressor` adalah estimator. Pemanggilan `fit()` menyimpan parameter yang dipelajari di dalam objek tersebut. `predict()` menggunakan parameter itu untuk memperkirakan target pada baris masukan baru.

Angka prediksi bukan jaminan nilai yang akan diperoleh. Model hanya mempelajari pola dari delapan contoh buatan di atas; kualitas prediksi pada kasus nyata bergantung pada mutu dan jumlah data yang representatif.

### `fit_predict()` pada pembelajaran tanpa label

Pada sejumlah algoritma unsupervised learning, kita tidak menyediakan target `y`. Contohnya clustering: algoritma membentuk kelompok berdasarkan kemiripan fitur. `KMeans.fit_predict()` mempelajari pusat kelompok dan sekaligus mengembalikan label kelompok untuk tiap baris.

In [ ]:
from sklearn.cluster import KMeans

# Contoh dua fitur pengukuran pada enam sampel.
data_sensor = np.array([
    [19.8, 5.1],
    [20.4, 5.3],
    [19.9, 4.9],
    [27.2, 7.6],
    [28.0, 7.9],
    [27.7, 8.1],
])

clusterer = KMeans(n_clusters=2, random_state=17, n_init=10)
label_cluster = clusterer.fit_predict(data_sensor)

print("Label cluster per sampel:", label_cluster)
print("Pusat cluster:")
print(clusterer.cluster_centers_)

Label cluster per sampel: [0 0 0 1 1 1]
Pusat cluster:
[[20.03333333  5.1       ]
 [27.63333333  7.86666667]]


Label `0` dan `1` hanyalah identitas kelompok, bukan peringkat atau kategori yang bermakna secara otomatis. Penomoran label dapat berubah saat data atau inisialisasi berubah; interpretasi seharusnya melihat pusat dan karakteristik masing-masing cluster.

## 4. Transformer dan `transform()`

Transformer digunakan untuk mengubah representasi fitur, misalnya dengan scaling atau encoding. Contoh di bawah memakai `StandardScaler`, yang menghitung rata-rata dan simpangan baku setiap fitur saat `fit()`, lalu menggunakan hasil itu saat `transform()`.

Scaling sering membantu algoritma yang sensitif terhadap skala fitur. Namun, standardisasi tidak berarti seluruh data menjadi berdistribusi normal.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fitur sengaja memiliki rentang nilai yang berbeda.
X_train_scale = np.array([
    [160, 42000],
    [172, 56000],
    [168, 51000],
    [181, 72000],
], dtype=float)

X_test_scale = np.array([
    [175, 61000],
    [158, 39000],
], dtype=float)

scaler = StandardScaler()
scaler.fit(X_train_scale)

X_train_scaled = scaler.transform(X_train_scale)
X_test_scaled = scaler.transform(X_test_scale)

print("Rata-rata yang dipelajari dari data latih:", scaler.mean_)
print("\nData latih setelah scaling:")
print(np.round(X_train_scaled, 3))
print("\nData uji setelah scaling dengan scaler yang sama:")
print(np.round(X_test_scaled, 3))

Rata-rata yang dipelajari dari data latih: [  170.25 55250.  ]

Data latih setelah scaling:
[[-1.355 -1.216]
 [ 0.231  0.069]
 [-0.298 -0.39 ]
 [ 1.422  1.537]]

Data uji setelah scaling dengan scaler yang sama:
[[ 0.628  0.528]
 [-1.62  -1.492]]


### Mengapa data uji tidak di-`fit()` ulang?

Transformasi untuk data uji perlu menggunakan parameter yang dipelajari dari data latih. Jika scaler di-fit ulang pada data uji, data uji akan menggunakan acuan berbeda dari model ketika dilatih. Ini dapat membuat evaluasi tidak mewakili proses penggunaan model sebenarnya.

Pola yang umum adalah `fit_transform()` pada data latih, kemudian `transform()` pada data validasi atau uji.

In [ ]:
scaler_alur_utama = StandardScaler()

# Satu langkah untuk mempelajari parameter dan mengubah data latih.
X_train_scaled_2 = scaler_alur_utama.fit_transform(X_train_scale)

# Data baru cukup diubah dengan parameter yang sudah dipelajari.
X_test_scaled_2 = scaler_alur_utama.transform(X_test_scale)

print("Hasil fit_transform konsisten dengan fit lalu transform:",
      np.allclose(X_train_scaled, X_train_scaled_2))
print("Hasil transform data uji konsisten:",
      np.allclose(X_test_scaled, X_test_scaled_2))

Hasil fit_transform konsisten dengan fit lalu transform: True
Hasil transform data uji konsisten: True


## 5. Membuat transformer sendiri

scikit-learn dapat diperluas melalui estimator dan transformer khusus. Agar dapat digunakan bersama komponen library lainnya, objek khusus perlu mengikuti pola API yang diharapkan.

Sebagai latihan, kita membuat `IQRClipper`. Transformer ini menghitung batas bawah dan batas atas berbasis rentang antarkuartil (IQR) dari data saat `fit()`, lalu membatasi nilai di luar batas itu ketika `transform()` dipanggil.

Implementasi ini untuk demonstrasi API; clipping bukan pilihan yang otomatis benar untuk semua dataset.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class IQRClipper(BaseEstimator, TransformerMixin):
    """Membatasi nilai fitur ke rentang berbasis IQR yang dipelajari dari data latih."""

    def __init__(self, factor=1.5):
        # Parameter konstruktor disimpan langsung agar cocok dengan pola estimator.
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X harus berupa array dua dimensi.")

        q1 = np.percentile(X, 25, axis=0)
        q3 = np.percentile(X, 75, axis=0)
        iqr = q3 - q1

        self.lower_bounds_ = q1 - self.factor * iqr
        self.upper_bounds_ = q3 + self.factor * iqr
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, ["lower_bounds_", "upper_bounds_"])
        X = np.asarray(X, dtype=float)

        if X.ndim != 2:
            raise ValueError("X harus berupa array dua dimensi.")
        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                f"Jumlah fitur saat transform harus {self.n_features_in_}, "
                f"bukan {X.shape[1]}."
            )

        return np.clip(X, self.lower_bounds_, self.upper_bounds_)

In [ ]:
data_pengamatan = np.array([
    [2.1, 10.0],
    [2.4, 11.0],
    [2.6, 10.5],
    [2.8, 12.0],
    [3.0, 11.5],
    [3.2, 10.8],
], dtype=float)

pengamatan_baru = np.array([
    [2.7, 10.9],
    [14.0, 95.0],  # nilai ekstrem buatan untuk memperlihatkan efek clipping
])

clipper = IQRClipper(factor=1.5)
clipper.fit(data_pengamatan)
hasil_clip = clipper.transform(pengamatan_baru)

print("Batas bawah:", np.round(clipper.lower_bounds_, 2))
print("Batas atas :", np.round(clipper.upper_bounds_, 2))
print("\nSebelum clipping:")
print(pengamatan_baru)
print("\nSesudah clipping:")
print(np.round(hasil_clip, 2))
print("\nParameter estimator:", clipper.get_params())

Batas bawah: [1.7  9.37]
Batas atas : [ 3.7  12.58]

Sebelum clipping:
[[ 2.7 10.9]
 [14.  95. ]]

Sesudah clipping:
[[ 2.7  10.9 ]
 [ 3.7  12.58]]

Parameter estimator: {'factor': 1.5}


Pada custom transformer tersebut, atribut yang diakhiri garis bawah—seperti `lower_bounds_`—menunjukkan nilai yang dipelajari saat `fit()`. Sementara itu, `factor` merupakan hyperparameter yang ditentukan pengguna. `check_is_fitted()` membantu mencegah `transform()` dipanggil sebelum objek dilatih.

## 6. Pipeline untuk menggabungkan preprocessing dan model

`Pipeline` menyusun beberapa tahap secara berurutan. Dalam contoh ini, data fitur distandardisasi lalu diberikan ke logistic regression. Dengan menyatukan dua tahap itu, kita dapat memanggil `fit()` dan `predict()` melalui satu objek.

Pipeline juga membantu mencegah *data leakage*: ketika proses evaluasi dilakukan dengan benar, scaler di-fit hanya pada bagian data latih di setiap pembagian data.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

dataset = load_breast_cancer()
X, y = dataset.data, dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=31, stratify=y
)

model_pipeline = Pipeline(steps=[
    ("standardisasi", StandardScaler()),
    ("klasifikasi", LogisticRegression(max_iter=3000, random_state=31)),
])

model_pipeline.fit(X_train, y_train)
prediksi_test = model_pipeline.predict(X_test)

print("Tahapan pipeline:", list(model_pipeline.named_steps.keys()))
print(f"Akurasi data latih: {model_pipeline.score(X_train, y_train):.3f}")
print(f"Akurasi data uji  : {accuracy_score(y_test, prediksi_test):.3f}")

Tahapan pipeline: ['standardisasi', 'klasifikasi']
Akurasi data latih: 0.988
Akurasi data uji  : 0.972


### Cara membaca hasil pipeline

Akurasi adalah proporsi prediksi yang tepat. Perbandingan akurasi latih dan uji dapat memberi petunjuk awal tentang kemampuan generalisasi, tetapi tidak cukup untuk menilai model dalam setiap situasi. Untuk data yang tidak seimbang atau masalah dengan biaya kesalahan berbeda, metrik tambahan seperti precision, recall, dan F1-score dapat lebih informatif.

`named_steps` menyediakan akses ke komponen pipeline berdasarkan nama, misalnya objek scaler melalui `model_pipeline.named_steps["standardisasi"]`.

## 7. Atribut dan metode umum

Banyak estimator menyediakan metode yang serupa, tetapi atribut yang tersedia setelah pelatihan bergantung pada jenis algoritmanya. Contoh umum:

- `get_params()` untuk melihat konfigurasi dan hyperparameter.
- `set_params()` untuk mengubah konfigurasi.
- `score()` untuk memperoleh skor default estimator.
- `n_features_in_` untuk melihat jumlah fitur yang diterima saat pelatihan.
- `coef_` dan `intercept_` tersedia pada model linear tertentu setelah fit.

Pada regresi, skor default `LinearRegression.score()` adalah \(R^2\), bukan persentase akurasi klasifikasi.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target

diabetes_model = LinearRegression()
diabetes_model.fit(X_diabetes, y_diabetes)

print("Jumlah fitur masukan:", diabetes_model.n_features_in_)
print("Lima koefisien pertama:", np.round(diabetes_model.coef_[:5], 3))
print("Intercept:", round(float(diabetes_model.intercept_), 3))
print(f"R-squared pada data contoh: {diabetes_model.score(X_diabetes, y_diabetes):.3f}")
print("Nama fitur pertama:", diabetes.feature_names[0])

Jumlah fitur masukan: 10
Lima koefisien pertama: [ -10.01  -239.816  519.846  324.385 -792.176]
Intercept: 152.133
R-squared pada data contoh: 0.518
Nama fitur pertama: age


## 8. Hyperparameter dan metode pencarian

Hyperparameter adalah pengaturan yang ditentukan sebelum proses belajar, contohnya kedalaman maksimum pohon atau jumlah pohon dalam random forest. Parameter yang dipelajari model dari data berbeda dari hyperparameter.

- `set_params()` mengubah konfigurasi estimator.
- `get_params()` menampilkan konfigurasi saat ini.
- `GridSearchCV` mencoba semua kombinasi pada grid dan membandingkannya dengan cross-validation.
- `RandomizedSearchCV` mengambil sejumlah kombinasi secara acak dari ruang parameter, sehingga dapat mengurangi jumlah percobaan untuk ruang pencarian yang besar.

Pemilihan hyperparameter dilakukan berdasarkan metrik dan strategi validasi yang ditentukan. Skor validasi bukan jaminan performa di semua data baru.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

rf_awal = RandomForestClassifier(random_state=23)
print("Sebelum diubah:", {
    "n_estimators": rf_awal.get_params()["n_estimators"],
    "max_depth": rf_awal.get_params()["max_depth"],
})

rf_awal.set_params(n_estimators=120, max_depth=5)
print("Sesudah diubah:", {
    "n_estimators": rf_awal.get_params()["n_estimators"],
    "max_depth": rf_awal.get_params()["max_depth"],
})

Sebelum diubah: {'n_estimators': 100, 'max_depth': None}
Sesudah diubah: {'n_estimators': 120, 'max_depth': 5}


In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=23, stratify=y_wine
)

grid_parameters = {
    "max_depth": [None, 3, 5, 7],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(n_estimators=100, random_state=23),
    param_grid=grid_parameters,
    cv=5,
    scoring="accuracy",
    n_jobs=1,
)
grid_search.fit(Xw_train, yw_train)

print("Parameter terbaik:", grid_search.best_params_)
print(f"Rata-rata skor cross-validation terbaik: {grid_search.best_score_:.3f}")
print(f"Akurasi pada data uji yang disisihkan: {grid_search.score(Xw_test, yw_test):.3f}")

Parameter terbaik: {'max_depth': None, 'min_samples_leaf': 1}
Rata-rata skor cross-validation terbaik: 0.970
Akurasi pada data uji yang disisihkan: 1.000


### Pencarian acak sebagai alternatif

Grid search menjadi mahal jika setiap hyperparameter memiliki banyak kandidat. Pencarian acak dapat menguji jumlah kombinasi yang dibatasi melalui `n_iter`. Hasilnya tidak harus sama dengan grid search karena kombinasi yang diuji berbeda.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_estimators=100, random_state=23),
    param_distributions={
        "max_depth": [None, 2, 3, 4, 5, 6, 8, 10],
        "min_samples_split": [2, 3, 4, 5, 8, 10],
        "min_samples_leaf": [1, 2, 3, 4],
    },
    n_iter=8,
    cv=5,
    scoring="accuracy",
    random_state=23,
    n_jobs=1,
)
random_search.fit(Xw_train, yw_train)

print("Parameter terpilih:", random_search.best_params_)
print(f"Skor cross-validation terbaik: {random_search.best_score_:.3f}")
print(f"Skor data uji: {random_search.score(Xw_test, yw_test):.3f}")

Parameter terpilih: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}
Skor cross-validation terbaik: 0.977
Skor data uji: 0.978


## 9. Metadata tags dan metadata routing

Estimator tags adalah informasi yang menjelaskan kemampuan serta karakteristik estimator kepada infrastruktur scikit-learn. Informasi tersebut dapat membantu fungsi library menentukan jenis target, format masukan, atau kapabilitas lain yang didukung oleh estimator.

Konsep yang berdekatan adalah *metadata routing*: informasi tambahan, misalnya sample weights atau group labels, dapat diarahkan kepada tahapan yang memang membutuhkannya. Cara melihat tags dapat berbeda antarversi scikit-learn, sehingga kode introspeksi di bawah memeriksa apakah antarmuka yang digunakan tersedia.

In [ ]:
tag_model = LogisticRegression(max_iter=1000)

if hasattr(tag_model, "__sklearn_tags__"):
    tags = tag_model.__sklearn_tags__()
    print("Jenis estimator:", tags.estimator_type)
    print("Memerlukan proses fit:", tags.requires_fit)
    print("Mendukung satu target:", tags.target_tags.single_output)
    print("Boleh menerima data sparse:", tags.input_tags.sparse)
    print("Mengizinkan nilai NaN:", tags.input_tags.allow_nan)
else:
    print(
        "Metode __sklearn_tags__ tidak tersedia pada versi ini. "
        "Periksa dokumentasi scikit-learn sesuai versi yang terpasang."
    )

Jenis estimator: classifier
Memerlukan proses fit: True
Mendukung satu target: True
Boleh menerima data sparse: True
Mengizinkan nilai NaN: False


Tags terutama berguna bagi komponen internal dan utilitas scikit-learn, bukan pengganti pemahaman terhadap dokumentasi model. Hindari bergantung pada detail internal yang dapat berubah antarversi tanpa memeriksa dokumentasi versi yang dipakai.

## 10. Praktik terbaik penggunaan API

Checklist untuk proyek machine learning yang mudah dipelihara:

1. Pisahkan fitur (`X`) dan target (`y`) dengan jelas.
2. Fit preprocessing hanya pada data latih; gunakan transformer yang telah fit untuk data validasi atau uji.
3. Gabungkan preprocessing dan model dalam `Pipeline` saat sesuai.
4. Gunakan cross-validation untuk menilai konfigurasi model, lalu evaluasi akhir pada data uji yang tidak ikut dipakai saat tuning.
5. Tetapkan `random_state` pada proses yang melibatkan randomness agar eksperimen lebih mudah diulang.
6. Gunakan nama langkah pipeline yang jelas dan dokumentasikan metrik yang dipakai.
7. Periksa bentuk data, jenis target, dan versi pustaka saat menemui masalah kompatibilitas.
8. Pilih metrik sesuai tujuan masalah, bukan hanya karena metrik tersebut mudah dihitung.

### Latihan mandiri

Coba modifikasi notebook ini sendiri sebelum mengumpulkan:

- Ubah pasangan fitur dan target pada contoh regresi, lalu amati perubahan prediksi.
- Bandingkan `StandardScaler` dengan `MinMaxScaler` pada data yang sama.
- Ubah `factor` pada `IQRClipper`, kemudian lihat bagaimana batas clipping berubah.
- Tambahkan satu kandidat `max_depth` ke parameter grid dan bandingkan hasil pencarian.
- Jelaskan mengapa scaler sebaiknya berada di dalam pipeline ketika dilakukan cross-validation.

Tuliskan hasil pengamatan dengan kata-kata sendiri; angka hasil dapat berbeda jika data atau konfigurasi diubah.

## Ringkasan Chapter 1

Chapter ini memperkenalkan pola API yang mendasari penggunaan scikit-learn:

| Konsep | Peran utama |
|---|---|
| Estimator | Mempelajari pola atau parameter melalui `fit()` |
| Predictor | Menghasilkan estimasi target melalui `predict()` |
| Transformer | Mengubah fitur melalui `transform()` |
| `fit_transform()` | Mempelajari transformasi dan menerapkannya dalam satu pemanggilan |
| `fit_predict()` | Melatih model dan mengembalikan label/hasil, misalnya pada clustering |
| Custom transformer | Memperluas API dengan komponen yang mengikuti pola metode scikit-learn |
| Pipeline | Menyatukan preprocessing dan estimator dalam satu alur |
| Atribut/metode umum | Membaca konfigurasi, atribut hasil fit, dan skor model |
| Hyperparameter search | Membandingkan pengaturan model dengan cross-validation |
| Tags/metadata | Menyatakan kemampuan estimator dan mendukung alur kerja library |

**Kesimpulan:** konsistensi dan modularitas API membuat komponen scikit-learn lebih mudah digabungkan, diuji, dan digunakan kembali. Keandalan eksperimen tetap bergantung pada pembagian data yang benar, pemilihan metrik yang sesuai, dan evaluasi yang tidak mengalami data leakage.

## Referensi

- Sukup, John. *scikit-learn Cookbook*, Third Edition. Packt Publishing, 2025. Chapter 1: “Common Conventions and API Elements of scikit-learn,” khususnya pembahasan design philosophy, estimators, transformers, custom estimators, pipelines, common attributes/methods, hyperparameter search, metadata tags, dan best practices.
- Repositori kode pendamping buku: https://github.com/PacktPublishing/scikit-learn-Cookbook-Third-Edition
- Dokumentasi scikit-learn: https://scikit-learn.org/stable/

Contoh pada notebook ini ditulis sebagai latihan implementasi dengan data dan struktur eksperimen tersendiri berdasarkan cakupan topik bab tersebut.